In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("Parkinsons-Telemonitoring-ucirvine.csv")

SEPARATE ROLES OF COLUMNS

In [4]:
data.columns

Index(['subject', 'age', 'sex', 'test_time', 'motor_updrs', 'total_updrs',
       'jitter', 'jitter_abs', 'jitter_rap', 'jitter_ppq5', 'jitter_ddp',
       'shimmer', 'shimmer_db', 'shimmer_apq3', 'shimmer_apq5',
       'shimmer_apq11', 'shimmer_dda', 'nhr', 'hnr', 'rpde', 'dfa', 'ppe'],
      dtype='object')

In [5]:
id_column = ['subject']
time_column = ['test_time']
target = ['motor_updrs', 'total_updrs']

meta_cols = [id_column, time_column, target, "sex", "age"]

SPLITTING THE DATA INTO TRAIN AND TEST DATA. SPLITTING BY PATIENTS, NOT BY ROWS

In [7]:
from sklearn.model_selection import GroupShuffleSplit

In [8]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next( gss.split(data, groups=data[id_column]))

train_data = data.iloc[train_idx]
test_data = data.iloc[test_idx]

In [9]:
# # Check if any subject appears in both sets
# train_subjects = set(train_data[id_column])
# test_subjects = set(test_data[id_column])
# common = train_subjects.intersection(test_subjects)
# print(f"Subjects in both sets: {common}")  # Should be empty!

In [10]:
train_data["sex"] = train_data["sex"].astype(int)
test_data["sex"]  = test_data["sex"].astype(int)

C:\Users\Durosimi\AppData\Local\Temp\ipykernel_5088\1486737828.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data["sex"] = train_data["sex"].astype(int)
C:\Users\Durosimi\AppData\Local\Temp\ipykernel_5088\1486737828.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data["sex"]  = test_data["sex"].astype(int)


In [11]:
train_data.to_csv("C://Users//Durosimi//PROJECTS//Parkinson/train_data.csv")

In [12]:
test_data.to_csv("C://Users//Durosimi//PROJECTS//Parkinson/test_data.csv")

In [13]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [14]:
selected_features = [
    "jitter",
    "shimmer",
    "dfa",
    "age",
    "sex"
]

In [15]:
X = train_data[selected_features]

In [16]:
X.dtypes

jitter     float64
shimmer    float64
dfa        float64
age          int64
sex          int32
dtype: object

In [17]:
X = X.apply(pd.to_numeric, errors="coerce")

In [18]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np
import pandas as pd

# Ensure all features are numeric floats
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna()

X_values = X.values.astype(float)

vif = pd.DataFrame()
vif["feature"] = X.columns
vif["VIF"] = [
    variance_inflation_factor(X_values, i)
    for i in range(X_values.shape[1])
]

vif


,feature,VIF
0,jitter,5.020847
1,shimmer,5.853509
2,dfa,39.015425
3,age,38.176298
4,sex,1.467251


In [19]:
from sklearn.preprocessing import StandardScaler
import joblib

In [20]:
#SCALING

In [21]:
scaler_linear = StandardScaler()

train_data[selected_features] = scaler_linear.fit_transform(
    train_data[selected_features]
)

test_data[selected_features] = scaler_linear.transform(
    test_data[selected_features]
)

joblib.dump(
    scaler_linear,
    "C:/Users/Durosimi/PROJECTS/Parkinson/models_artifacts/scaler_linear.pkl"
)

C:\Users\Durosimi\AppData\Local\Temp\ipykernel_5088\2051907703.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[selected_features] = scaler_linear.fit_transform(
C:\Users\Durosimi\AppData\Local\Temp\ipykernel_5088\2051907703.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data[selected_features] = scaler_linear.transform(


['C:/Users/Durosimi/PROJECTS/Parkinson/models_artifacts/scaler_linear.pkl']

In [22]:
#SAVE LINEAR FEATURES

In [23]:
linear_cols = selected_features + target

train_data[linear_cols].to_csv(
    "C:/Users/Durosimi/PROJECTS/Parkinson/data/features/train_linear.csv",
    index=False
)

test_data[linear_cols].to_csv(
    "C:/Users/Durosimi/PROJECTS/Parkinson/data/features/test_linear.csv",
    index=False
)

In [24]:
tree_features = [
    "jitter",
    "jitter_ddp",
    "shimmer",
    "shimmer_dda",
    "hnr",
    "rpde",
    "dfa",
    "ppe",
    "age",
    "sex"
]

In [25]:
scaler_tree = StandardScaler()

train_data[tree_features] = scaler_tree.fit_transform(
    train_data[tree_features]
)

test_data[tree_features] = scaler_tree.transform(
    test_data[tree_features]
)

joblib.dump(
    scaler_tree,
    "C:/Users/Durosimi/PROJECTS/Parkinson/models_artifacts/scaler_tree.pkl"
)

C:\Users\Durosimi\AppData\Local\Temp\ipykernel_5088\850547040.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data[tree_features] = scaler_tree.fit_transform(
C:\Users\Durosimi\AppData\Local\Temp\ipykernel_5088\850547040.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_data[tree_features] = scaler_tree.transform(


['C:/Users/Durosimi/PROJECTS/Parkinson/models_artifacts/scaler_tree.pkl']

In [26]:
tree_cols = tree_features + target

train_data[tree_cols].to_csv(
    "C:/Users/Durosimi/PROJECTS/Parkinson/data/features/train_tree.csv",
    index=False
)

test_data[tree_cols].to_csv(
    "C:/Users/Durosimi/PROJECTS/Parkinson/data/features/test_tree.csv",
    index=False
)

In [63]:
import os

OUTPUT_PATH = "C:/Users/Durosimi/PROJECTS/Parkinson/data/processed"
os.makedirs(OUTPUT_PATH, exist_ok=True)

data.to_csv(
    os.path.join(OUTPUT_PATH, "featured_data.csv"),
    index=False
)

print("✅ featured_data.csv saved to data/processed/")


✅ featured_data.csv saved to data/processed/


In [65]:
df = pd.read_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/featured_data.csv")

In [67]:
df.head()

,subject,age,sex,test_time,motor_updrs,total_updrs,jitter,jitter_abs,jitter_rap,jitter_ppq5,...,shimmer_db,shimmer_apq3,shimmer_apq5,shimmer_apq11,shimmer_dda,nhr,hnr,rpde,dfa,ppe
0,1,72,False,5.6431,28.199,34.398,0.00662,0.000034,0.00401,0.00317,...,0.230,0.01438,0.01309,0.01662,0.04314,0.014290,21.640,0.41888,0.54842,0.16006
1,1,72,False,12.6660,28.447,34.894,0.00300,0.000017,0.00132,0.00150,...,0.179,0.00994,0.01072,0.01689,0.02982,0.011112,27.183,0.43493,0.56477,0.10810
2,1,72,False,19.6810,28.695,35.389,0.00481,0.000025,0.00205,0.00208,...,0.181,0.00734,0.00844,0.01458,0.02202,0.020220,23.047,0.46222,0.54405,0.21014
3,1,72,False,25.6470,28.905,35.810,0.00528,0.000027,0.00191,0.00264,...,0.327,0.01106,0.01265,0.01963,0.03317,0.027837,24.445,0.48730,0.57794,0.33277
4,1,72,False,33.6420,29.187,36.375,0.00335,0.000020,0.00093,0.00130,...,0.176,0.00679,0.00929,0.01819,0.02036,0.011625,26.126,0.47188,0.56122,0.19361


### Feature Engineering Summary

- Patient-level group split applied to avoid leakage
- Severe multicollinearity detected in EDA
- Linear models use pruned, interpretable feature set
- Tree-based models retain full acoustic feature space
- Features standardized using StandardScaler
- Separate datasets saved for each modeling family

This setup supports robust benchmarking, interpretability,
and production deployment.